# BOQ Data Walkthrough — Client Presentation

**Purpose:** answer, with real numbers from TAQA's actual Maximo data (Unity Catalog: `ingestion_framework_test.bid_data_exploration`), the questions that determine how the bid-analyzer comparison pipeline should be built against real data rather than the hand-curated sample set. Full technical background: `databricks/FINDINGS.md`.

**Run this top to bottom in Databricks** — every chart and number below is computed live from the real tables, nothing is hardcoded.

1. Of all RFQs/bids, how many actually have a detailed, line-by-line BOQ?
2. For the ones that do — how are negotiation rounds tracked?
3. For the ones that don't — what does the data actually look like?
4. Four open questions to confirm with TAQA before building further.

## Setup

In [ ]:
CATALOG = "ingestion_framework_test"
SCHEMA = "bid_data_exploration"
FQ = f"{CATALOG}.{SCHEMA}"

import pandas as pd
import plotly.graph_objects as go

pd.set_option("display.max_colwidth", 120)

# Categorical palette + chart chrome (validated 8-hue order; see the
# Claude Code dataviz skill's references/palette.md -- reused as-is,
# not brand-customized here). Sequential blue is used for magnitude-only
# comparisons; categorical slots are assigned in fixed order, never by
# perceived meaning (no red-for-bad / green-for-good).
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ_BLUE = {100: "#cde2fb", 250: "#86b6ef", 400: "#3987e5", 550: "#1c5cab", 700: "#0d366b"}
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRID = "#e1e0d9"
SURFACE = "#fcfcfb"


def style_fig(fig, title, height=440, showlegend=False):
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK_PRIMARY)),
        plot_bgcolor=SURFACE,
        paper_bgcolor=SURFACE,
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color=INK_SECONDARY, size=12),
        height=height,
        margin=dict(l=70, r=30, t=60, b=60),
        showlegend=showlegend,
        legend=dict(bgcolor="rgba(0,0,0,0)", orientation="h", y=1.1, x=0),
        xaxis=dict(showgrid=False, linecolor=GRID, tickfont=dict(color=INK_MUTED)),
        yaxis=dict(showgrid=True, gridcolor=GRID, zeroline=False, tickfont=dict(color=INK_MUTED)),
    )
    return fig


def aed(x):
    """Format a raw AED number as a compact millions string for display."""
    if x is None or pd.isna(x):
        return "n/a"
    return f"AED {x/1e6:,.1f}M"

print("Setup complete.")

## 1. Of all RFQs/bids, how many actually have a detailed, line-by-line BOQ?

**Why not just use `rfq.DETAILBOQAVAILABLE`?** We tested this flag directly (`FINDINGS.md`) and found it unreliable: D-111808 — a real, fully awarded 142.46M construction tender — has this flag set to `null`, not `Y` or `N`. Treating `null` as "no BOQ" would be wrong. Instead, this section measures the **real structural signal**: how many priced line items does `quotationline` actually have per vendor, for each RFQ? That tells us directly whether a tender was priced as one lump sum or broken into real line items — regardless of what any flag says.

In [ ]:
boq_shape_df = spark.sql(f"""
    WITH vendor_lines AS (
        SELECT RFQNUM, VENDOR, COUNT(*) AS line_count
        FROM {FQ}.quotationline
        GROUP BY RFQNUM, VENDOR
    ),
    rfq_lines AS (
        SELECT RFQNUM,
               AVG(line_count) AS avg_lines_per_vendor,
               MAX(line_count) AS max_lines_per_vendor,
               COUNT(DISTINCT VENDOR) AS vendor_count
        FROM vendor_lines
        GROUP BY RFQNUM
    )
    SELECT r.RFQNUM, r.DESCRIPTION, r.TOTALAWVALUE, r.DETAILBOQAVAILABLE,
           rl.avg_lines_per_vendor, rl.max_lines_per_vendor, rl.vendor_count
    FROM {FQ}.rfq r
    LEFT JOIN rfq_lines rl ON r.RFQNUM = rl.RFQNUM
""").toPandas()

def categorize(avg_lines):
    if pd.isna(avg_lines):
        return "No priced lines at all"
    if avg_lines <= 1:
        return "Lump-sum (1 line/vendor)"
    if avg_lines < 10:
        return "Shallow (2-9 lines/vendor)"
    return "Detailed BOQ (10+ lines/vendor)"

boq_shape_df["category"] = boq_shape_df["avg_lines_per_vendor"].apply(categorize)

summary = (
    boq_shape_df.groupby("category")
    .agg(
        rfq_count=("RFQNUM", "count"),
        total_award_value=("TOTALAWVALUE", "sum"),
        rfqs_with_award_value=("TOTALAWVALUE", lambda s: s.notna().sum()),
    )
    .reset_index()
)
summary["pct_of_all_rfqs"] = (summary["rfq_count"] / len(boq_shape_df) * 100).round(1)
summary = summary.sort_values("rfq_count", ascending=False)

print(f"Total RFQs in scope: {len(boq_shape_df):,}")
summary

**Reading this table:** `total_award_value` only sums RFQs where `TOTALAWVALUE` is actually populated at the header level — most rows have it `null` (award totals often live at the vendor/line level instead, see Section 4c), so treat the value breakdown as directional, not comprehensive. `rfqs_with_award_value` shows how many RFQs in each bucket actually had a value to sum.

In [ ]:
order = [
    "Detailed BOQ (10+ lines/vendor)",
    "Shallow (2-9 lines/vendor)",
    "Lump-sum (1 line/vendor)",
    "No priced lines at all",
]
plot_df = summary.set_index("category").reindex(order).reset_index()

fig = go.Figure(go.Bar(
    x=plot_df["category"], y=plot_df["rfq_count"],
    marker_color=CAT[:4],
    text=[f"{n:,}<br>({p:.1f}%)" for n, p in zip(plot_df["rfq_count"], plot_df["pct_of_all_rfqs"])],
    textposition="outside",
))
style_fig(fig, "How many RFQs have a real, detailed BOQ? (by line count per vendor)")
fig.update_yaxes(title_text="Number of RFQs")
fig.show()

In [ ]:
value_df = plot_df[plot_df["total_award_value"].notna()]
fig = go.Figure(go.Bar(
    x=value_df["category"], y=value_df["total_award_value"] / 1e6,
    marker_color=CAT[:len(value_df)],
    text=[f"AED {v/1e6:,.0f}M" for v in value_df["total_award_value"]],
    textposition="outside",
))
style_fig(fig, "Total recorded award value by BOQ category (AED millions, where TOTALAWVALUE is populated)")
fig.update_yaxes(title_text="AED millions")
fig.show()

### Does `DETAILBOQAVAILABLE` actually line up with this?

Short answer, already established in `FINDINGS.md`: not reliably. Shown here again against the real structural categories above, side by side, as the direct evidence for that claim.

In [ ]:
cross = pd.crosstab(boq_shape_df["category"], boq_shape_df["DETAILBOQAVAILABLE"].fillna("null"))
cross = cross.reindex(order)
cross

In [ ]:
fig = go.Figure()
for i, flag_val in enumerate(cross.columns):
    fig.add_trace(go.Bar(
        name=str(flag_val), x=cross.index, y=cross[flag_val],
        marker_color=CAT[i],
    ))
fig.update_layout(barmode="stack")
style_fig(fig, "DETAILBOQAVAILABLE flag vs. real BOQ structure — the flag doesn't predict the category", showlegend=True)
fig.update_yaxes(title_text="Number of RFQs")
fig.show()

## 2. For tenders WITH a detailed BOQ — how do we track negotiation rounds?

Picking the richest real example from the "Detailed BOQ" bucket above to walk through concretely.

In [ ]:
richest = spark.sql(f"""
    SELECT r.RFQNUM, r.DESCRIPTION, r.TOTALAWVALUE, r.DISCOUNT_REVISION,
           COUNT(ql.QUOTATIONLINEID) AS line_count,
           COUNT(DISTINCT ql.VENDOR) AS vendor_count
    FROM {FQ}.rfq r
    JOIN {FQ}.quotationline ql ON r.RFQNUM = ql.RFQNUM
    GROUP BY r.RFQNUM, r.DESCRIPTION, r.TOTALAWVALUE, r.DISCOUNT_REVISION
    HAVING COUNT(ql.QUOTATIONLINEID) / COUNT(DISTINCT ql.VENDOR) >= 10
    ORDER BY line_count DESC
    LIMIT 10
""").toPandas()
richest

**Pick one `RFQNUM` from the table above and set it below** — every cell in this section re-runs against whichever one you choose.

In [ ]:
EXAMPLE_RFQNUM = richest.iloc[0]["RFQNUM"] if len(richest) else None
# EXAMPLE_RFQNUM = "N-19535"  # uncomment and edit to pick a specific one instead
print("Walking through:", EXAMPLE_RFQNUM)

In [ ]:
sample_lines = spark.sql(f"""
    SELECT VENDOR, RFQLINENUM, BOQITEMNUM, DESCRIPTION, ORDERQTY, ORDERUNIT,
           UNITCOST, LINECOST, LINECOSTWDIS, DISCOUNT_PERCENT, LINETYPE, ISAWARDED
    FROM {FQ}.quotationline
    WHERE RFQNUM = '{EXAMPLE_RFQNUM}'
    ORDER BY VENDOR, RFQLINENUM
    LIMIT 25
""").toPandas()
sample_lines

**What to point out here to the client:** no separate CIF/Erection columns, and `BOQITEMNUM` is null for most real detailed BOQs (confirmed on the largest example we found, 2,480 lines, all null) — the real line-item identity is `RFQLINENUM`, consistent across every vendor for the same tender (verified: 100% match on `DESCRIPTION`/`ORDERQTY`/`ORDERUNIT` across vendors on a 620-line real example).

### The round-tracking mechanism: header-level counter + line-level before/after

`rfq.DISCOUNT_REVISION` (and `rfqvendor.POSTBID_DISCOUNT_COUNTER`, which moves in lockstep with it) is the closest thing to a "which round are we on" counter. Distribution across all RFQs:

In [ ]:
rev_dist = spark.sql(f"""
    SELECT DISCOUNT_REVISION, COUNT(*) AS n
    FROM {FQ}.rfq
    WHERE DISCOUNT_REVISION IS NOT NULL
    GROUP BY DISCOUNT_REVISION
    ORDER BY DISCOUNT_REVISION
""").toPandas()

fig = go.Figure(go.Bar(
    x=rev_dist["DISCOUNT_REVISION"].astype(int), y=rev_dist["n"],
    marker_color=SEQ_BLUE[400],
    text=rev_dist["n"], textposition="outside",
))
style_fig(fig, "How many RFQs reach each negotiation round? (rfq.DISCOUNT_REVISION)")
fig.update_xaxes(title_text="Round number", dtick=1)
fig.update_yaxes(title_text="Number of RFQs")
fig.show()

At the **line-item level**, the round's *effect* is visible as `LINECOST` (original quote) vs. `LINECOSTWDIS` (after discount) — but there's no round-by-round history, only original vs. final. Sample from our walkthrough tender, lines where a discount was actually applied:

In [ ]:
# Dedicated query (not derived from the 25-row sample above) so this
# chart has real content regardless of what happened to be in that
# earlier LIMIT -- filters directly for lines where a discount actually
# applied, on the full tender, then takes the highest-value examples.
discounted = spark.sql(f"""
    SELECT VENDOR, RFQLINENUM, LINECOST, LINECOSTWDIS
    FROM {FQ}.quotationline
    WHERE RFQNUM = '{EXAMPLE_RFQNUM}'
      AND LINECOST IS NOT NULL AND LINECOSTWDIS IS NOT NULL
      AND LINECOST != LINECOSTWDIS
    ORDER BY LINECOST DESC
    LIMIT 12
""").toPandas()

if len(discounted):
    labels = [f"{v} · line {int(l)}" for v, l in zip(discounted["VENDOR"], discounted["RFQLINENUM"])]
    fig = go.Figure()
    for lab, before, after in zip(labels, discounted["LINECOST"], discounted["LINECOSTWDIS"]):
        fig.add_trace(go.Scatter(
            x=[before, after], y=[lab, lab], mode="lines",
            line=dict(color=INK_MUTED, width=2), showlegend=False,
        ))
    fig.add_trace(go.Scatter(
        x=discounted["LINECOST"], y=labels, mode="markers",
        name="Original quote", marker=dict(color=SEQ_BLUE[700], size=11),
    ))
    fig.add_trace(go.Scatter(
        x=discounted["LINECOSTWDIS"], y=labels, mode="markers",
        name="After discount", marker=dict(color=SEQ_BLUE[250], size=11),
    ))
    style_fig(fig, "Original vs. discounted price, per line item (sample)", height=460, showlegend=True)
    fig.update_xaxes(title_text="AED")
    fig.show()
else:
    print("No discounted lines in this particular sample -- re-run with a different EXAMPLE_RFQNUM above.")

**Key caveat to raise with the client**: `quotationline` has no round-number column and no change-timestamp — only `ENTERDATE` (a single initial-entry stamp). So this view only ever shows **original vs. final state**. If there were 3 intermediate negotiation rounds, they're overwritten, not preserved. This is exactly the gap Question 4b (below) asks TAQA to resolve.

## 3. For tenders WITHOUT a detailed BOQ — what does the data actually look like?

Two real examples: **D-111808** (the exact tender the original sample data is modeled on) and one more picked live from the "Lump-sum" bucket.

In [ ]:
d111808 = spark.sql(f"""
    SELECT VENDOR, RFQLINENUM, BOQITEMNUM, DESCRIPTION, ORDERQTY, ORDERUNIT,
           UNITCOST, LINECOST, LINETYPE, ISAWARDED
    FROM {FQ}.quotationline
    WHERE RFQNUM = 'D-111808'
    ORDER BY LINECOST DESC
""").toPandas()
d111808

**One row per vendor, for the entire tender** — `DESCRIPTION` repeats the whole-tender description, not a BOQ item; `BOQITEMNUM` is null; `ORDERQTY` is always `1`. This is a lump-sum bid, structurally identical to a single line item, not an itemized BOQ broken down by the sample data's ~218-rows-per-lot model.

In [ ]:
second_example_candidates = spark.sql(f"""
    WITH vendor_lines AS (
        SELECT RFQNUM, VENDOR, COUNT(*) AS line_count
        FROM {FQ}.quotationline
        GROUP BY RFQNUM, VENDOR
    ),
    rfq_lines AS (
        SELECT RFQNUM, AVG(line_count) AS avg_lines_per_vendor, COUNT(DISTINCT VENDOR) AS vendor_count
        FROM vendor_lines GROUP BY RFQNUM
    )
    SELECT r.RFQNUM, r.DESCRIPTION, r.TOTALAWVALUE, rl.vendor_count
    FROM {FQ}.rfq r
    JOIN rfq_lines rl ON r.RFQNUM = rl.RFQNUM
    WHERE rl.avg_lines_per_vendor <= 1 AND rl.vendor_count >= 3 AND r.TOTALAWVALUE IS NOT NULL
    ORDER BY r.TOTALAWVALUE DESC
    LIMIT 10
""").toPandas()
second_example_candidates

In [ ]:
# SECOND_EXAMPLE = second_example_candidates.iloc[0]["RFQNUM"]  # or set a specific one from the table above
SECOND_EXAMPLE = second_example_candidates.iloc[0]["RFQNUM"] if len(second_example_candidates) else None

second_example_lines = spark.sql(f"""
    SELECT VENDOR, RFQLINENUM, BOQITEMNUM, DESCRIPTION, ORDERQTY, ORDERUNIT,
           UNITCOST, LINECOST, LINETYPE, ISAWARDED
    FROM {FQ}.quotationline
    WHERE RFQNUM = '{SECOND_EXAMPLE}'
    ORDER BY LINECOST DESC
""").toPandas()
print("Second lump-sum example:", SECOND_EXAMPLE)
second_example_lines

## 4. Four questions to confirm with TAQA before building further

Each one below is presented with the actual evidence we have, and the specific gap that only TAQA can close.

### 4a. Where do we find vendor names?

`rfqvendor.VENDOR` is a bare code (e.g. `001938`, `99473989`), not a company name. We've searched every column across all 7 tables/views in this schema for anything name-like — here's the live result of that search:

In [ ]:
name_cols = spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM {CATALOG}.information_schema.columns
    WHERE table_schema = '{SCHEMA}'
      AND (column_name ILIKE '%NAME%' OR column_name ILIKE '%VENDOR%' OR column_name ILIKE '%COMPANY%')
    ORDER BY table_name, column_name
""").toPandas()
name_cols

None of these resolve a vendor **code** to a company name (`CONTACT` is a person's name at the vendor, `MANUFACTURERNAME` is a product manufacturer, not the bidder). **Question for TAQA: is there a vendor/company master table (a Maximo `COMPANIES`-style table) we haven't been given access to yet?**

### 4b. Where do we find round revisions, with history tracking?

Covered in Section 2: `DISCOUNT_REVISION`/`POSTBID_DISCOUNT_COUNTER` tell us **how many** rounds happened, and `LINECOST`/`LINECOSTWDIS` show the **net effect**, but there's no column anywhere that preserves what was quoted after round 1 vs. round 2 vs. round 3 individually — only the original and the current/final state. **Question for TAQA: does that intermediate round-by-round history exist anywhere in Maximo (even outside these 7 tables), or is it genuinely not retained?**

### 4c. Is `TOTALAWARDCOSTWDIS` really the confirmed final price?

Testing this directly: for every case where a vendor's `TOTALAWARDCOSTWDIS` is populated and non-zero, does it match the RFQ header's `TOTALAWVALUE`?

In [ ]:
wdis_check = spark.sql(f"""
    SELECT rv.RFQNUM, r.TOTALAWVALUE AS rfq_header_award_value,
           rv.VENDOR, rv.TOTALAWARDCOSTWDIS AS vendor_wdis_value,
           ABS(r.TOTALAWVALUE - rv.TOTALAWARDCOSTWDIS) < 1 AS values_match
    FROM {FQ}.rfqvendor rv
    JOIN {FQ}.rfq r ON rv.RFQNUM = r.RFQNUM
    WHERE rv.TOTALAWARDCOSTWDIS IS NOT NULL AND rv.TOTALAWARDCOSTWDIS > 0
      AND r.TOTALAWVALUE IS NOT NULL
    ORDER BY r.TOTALAWVALUE DESC
    LIMIT 15
""").toPandas()
wdis_check

In [ ]:
if len(wdis_check):
    match_rate = wdis_check["values_match"].mean() * 100
    print(f"Match rate on this sample: {match_rate:.0f}% ({wdis_check['values_match'].sum()}/{len(wdis_check)})")

    plot_df = wdis_check.head(8)
    labels = plot_df["RFQNUM"] + " · " + plot_df["VENDOR"]
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="rfq.TOTALAWVALUE (header)", x=labels, y=plot_df["rfq_header_award_value"] / 1e6,
        marker_color=CAT[0],
    ))
    fig.add_trace(go.Bar(
        name="rfqvendor.TOTALAWARDCOSTWDIS", x=labels, y=plot_df["vendor_wdis_value"] / 1e6,
        marker_color=CAT[1],
    ))
    fig.update_layout(barmode="group")
    style_fig(fig, "Header award value vs. vendor WDIS value -- do they match?", height=460, showlegend=True)
    fig.update_yaxes(title_text="AED millions")
    fig.show()
else:
    print("No rows with both values populated in this sample -- widen the LIMIT above if needed.")

If the match rate above is at or near 100%, that's strong evidence `TOTALAWARDCOSTWDIS` ("with discount") is genuinely the final confirmed award price. **Question for TAQA: please confirm this is the intended field to treat as the authoritative final price** — and if so, whether `TOTALAWARDCOSTWITHTAXWDIS` (same figure plus tax) is the one that should actually drive comparisons, given UAE VAT.

### 4d. What is `altquotationline`?

Schema is near-identical to `quotationline`, plus `ALTQUOTATIONLINEID` (its own PK) and `ALTQUOTLINEUID`. Live row-count comparison:

In [ ]:
alt_vs_base = spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {FQ}.altquotationline) AS alt_line_count,
        (SELECT COUNT(*) FROM {FQ}.quotationline) AS base_line_count
""").toPandas()

alt_n, base_n = alt_vs_base.iloc[0]["alt_line_count"], alt_vs_base.iloc[0]["base_line_count"]
fig = go.Figure(go.Bar(
    x=["quotationline (base lines)", "altquotationline (alternates)"],
    y=[base_n, alt_n],
    marker_color=[SEQ_BLUE[550], SEQ_BLUE[250]],
    text=[f"{base_n:,}", f"{alt_n:,}"], textposition="outside",
))
style_fig(fig, f"Alternates are rare: {alt_n/base_n*100:.1f}% of base line volume")
fig.update_yaxes(type="log", title_text="Row count (log scale)")
fig.show()

We tested whether `ALTQUOTLINEUID` joins back to `quotationline.QUOTATIONLINEID` (the theory: "this is an alternate *for* that specific base line") — it returned **zero matching rows**, so that theory is disproved. `QL2` (a generic extension column, shared with `quotationline`) holds values like `QUOTED`/`TNA`/`NOQUOTE`/`Cancel`/`CNA` — reads like a technical-acceptance status, but that's inferred from values, not confirmed. **Question for TAQA: what is `altquotationline` actually used for operationally, and what does `ALTQUOTLINEUID` really reference?**

## Summary for the walkthrough

- **Section 1** gives the headline split (detailed / shallow / lump-sum / no pricing data) with counts, percentages, and value where available — the number to lead with.
- **Section 2** shows a real detailed BOQ and exactly how negotiation rounds show up in the data (header counter + line-level before/after, no in-between history).
- **Section 3** shows two real lump-sum examples, D-111808 among them, side by side with the detailed case for contrast.
- **Section 4** is the actual ask: four specific, evidenced questions TAQA needs to answer before the comparison pipeline can be built with confidence.